In [2]:
!pip install torch torchvision

   ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.2 MB ? eta -:--:--
   ----- ---------------------------------- 0.8/6.2 MB 2.1 MB/s eta 0:00:03
   ------ --------------------------------- 1.0/6.2 MB 1.9 MB/s eta 0:00:03
   ---------- ----------------------------- 1.6/6.2 MB 2.0 MB/s eta 0:00:03
   --------------- ------------------------ 2.4/6.2 MB 2.3 MB/s eta 0:00:02
   ------------------ --------------------- 2.9/6.2 MB 2.4 MB/s eta 0:00:02
   ----------------------- ---------------- 3.7/6.2 MB 2.5 MB/s eta 0:00:01
   ---------------------------- ----------- 4.5/6.2 MB 2.8 MB/s eta 0:00:01
   ----------------------------------- ---- 5.5/6.2 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 6.2/6.2 MB 3.1 MB/s  0:00:02

  Attempting uninstall: sympy

    Found existing installation: sympy 1.14.0

   ------------------------------

In [3]:
import os
import torch
import torch.nn as nn
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
import numpy as np

learning_rate  = 1e-4
keep_prob_rate = 0.7
max_epoch      = 3
BATCH_SIZE     = 50

# 下载数据
DOWNLOAD_MNIST = not (os.path.exists('./mnist/') and os.listdir('./mnist/'))

train_data = torchvision.datasets.MNIST(
    root='./mnist/', train=True,
    transform=torchvision.transforms.ToTensor(),
    download=DOWNLOAD_MNIST,
)
train_loader = Data.DataLoader(
    dataset=train_data, batch_size=BATCH_SIZE, shuffle=True
)

test_data = torchvision.datasets.MNIST(root='./mnist/', train=False)

with torch.no_grad():
    test_x = torch.unsqueeze(test_data.data, dim=1).type(torch.FloatTensor)[:500] / 255.
test_y = test_data.targets[:500].numpy()  #  test_labels → targets


class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32,
                      kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64,
                      kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.out1    = nn.Linear(7 * 7 * 64, 1024, bias=True)
        self.dropout = nn.Dropout(keep_prob_rate)
        self.out2    = nn.Linear(1024, 10, bias=True)

    def forward(self, x):
        x   = self.conv1(x)
        x   = self.conv2(x)
        x   = x.view(-1, 7 * 7 * 64)
        out = self.out1(x)
        out = F.relu(out)
        out = self.dropout(out)
        out = self.out2(out)
        return F.softmax(out, dim=1)  


def test(cnn):
    cnn.eval()  #   切换到评估模式，关闭 dropout
    with torch.no_grad():
        y_pre     = cnn(test_x)
        _, pre_index = torch.max(y_pre, 1)
        prediction   = pre_index.numpy()
    cnn.train()  # 测完切回训练模式
    return np.sum(prediction == test_y) / 500.0


def train(cnn):
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate)
    loss_func = nn.CrossEntropyLoss()
    for epoch in range(max_epoch):
        for step, (x_, y_) in enumerate(train_loader):
            output = cnn(x_)
            loss   = loss_func(output, y_)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if step != 0 and step % 20 == 0:
                print(f"Epoch {epoch} | Step {step} | 测试准确率: {test(cnn):.4f}")


#：Jupyter里直接调用，不用 if __name__ == '__main__'
cnn = CNN()
train(cnn)

Epoch 0 | Step 20 | 测试准确率: 0.3520
Epoch 0 | Step 40 | 测试准确率: 0.4560
Epoch 0 | Step 60 | 测试准确率: 0.6660
Epoch 0 | Step 80 | 测试准确率: 0.6220
Epoch 0 | Step 100 | 测试准确率: 0.6880
Epoch 0 | Step 120 | 测试准确率: 0.7280
Epoch 0 | Step 140 | 测试准确率: 0.7340
Epoch 0 | Step 160 | 测试准确率: 0.8180
Epoch 0 | Step 180 | 测试准确率: 0.8460
Epoch 0 | Step 200 | 测试准确率: 0.8500
Epoch 0 | Step 220 | 测试准确率: 0.8560
Epoch 0 | Step 240 | 测试准确率: 0.8840
Epoch 0 | Step 260 | 测试准确率: 0.8900
Epoch 0 | Step 280 | 测试准确率: 0.9000
Epoch 0 | Step 300 | 测试准确率: 0.9060
Epoch 0 | Step 320 | 测试准确率: 0.9140
Epoch 0 | Step 340 | 测试准确率: 0.9040
Epoch 0 | Step 360 | 测试准确率: 0.9160
Epoch 0 | Step 380 | 测试准确率: 0.9160
Epoch 0 | Step 400 | 测试准确率: 0.9160
Epoch 0 | Step 420 | 测试准确率: 0.9140
Epoch 0 | Step 440 | 测试准确率: 0.9120
Epoch 0 | Step 460 | 测试准确率: 0.9180
Epoch 0 | Step 480 | 测试准确率: 0.9240
Epoch 0 | Step 500 | 测试准确率: 0.9160
Epoch 0 | Step 520 | 测试准确率: 0.9320
Epoch 0 | Step 540 | 测试准确率: 0.9240
Epoch 0 | Step 560 | 测试准确率: 0.9360
Epoch 0 | Step 580 | 测试准